In [1]:
!pip install drain3 pandas scikit-learn

  Preparing metadata (setup.py) ... done
  Created wheel for drain3: filename=drain3-0.9.11-py3-none-any.whl size=23998 sha256=a40ee4d2b35b06a5278fa11f07c25a56350d3247b9f01217c09ec7c647a73979
  Stored in directory: /root/.cache/pip/wheels/3f/d1/46/58e1747b3d77c4990f838e1c1f610f5aab1a21889cc9bff5c2
Successfully built drain3
  Attempting uninstall: jsonpickle
    Found existing installation: jsonpickle 4.1.1
    Uninstalling jsonpickle-4.1.1:
      Successfully uninstalled jsonpickle-4.1.1
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.6
    Uninstalling cachetools-6.2.6:
      Successfully uninstalled cachetools-6.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires cachetools<7.0,>=5.5, but you have cachetools 4.2.1 which is incompatible.


Phase 1: Parse Log với Drain3

In [7]:
import os
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

log_file = "/content/drive/MyDrive/AIOps/w1/d2/BGL_2k.log"

# Load log file, đếm tổng số dòng
total_raw_lines = 0
with open(log_file, 'r', encoding='utf-8') as f:
    total_raw_lines = sum(1 for line in f)
print(f"1. Tổng số dòng trong file log gốc: {total_raw_lines:,} dòng\n")

# Tune drain_sim_th (0.3, 0.5, 0.7)
print("2. TUNING drain_sim_th")
sim_ths = [0.3, 0.5, 0.7]
for th in sim_ths:
    config = TemplateMinerConfig()
    config.drain_sim_th = th
    miner_tune = TemplateMiner(config=config)

    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            miner_tune.add_log_message(line.strip())

    print(f"   - Ngưỡng sim_th = {th} -> Số templates tạo ra: {len(miner_tune.drain.clusters)}")

# Parse toàn bộ log với Drain3
# Liệt kê TẤT CẢ templates
print("3. PARSE LOG VỚI sim_th = 0.5")
config = TemplateMinerConfig()
config.drain_sim_th = 0.5
miner = TemplateMiner(config=config)

with open(log_file, 'r', encoding='utf-8') as f:
    for line in f:
        miner.add_log_message(line.strip())

clusters = miner.drain.clusters
print(f"   -> Đã parse xong! Tạo ra được {len(clusters)} templates unique.\n")

# Đưa tất cả templates vào DataFrame để dễ liệt kê và đếm số dòng
all_templates = []
for c in clusters:
    all_templates.append({
        'template_id': f"T-{c.cluster_id}",
        'template': c.get_template(),
        'count': c.size
    })

df_all = pd.DataFrame(all_templates)
# Sắp xếp giảm dần
df_all = df_all.sort_values(by='count', ascending=False).reset_index(drop=True)

print("4. LIỆT KÊ TẤT CẢ TEMPLATES")
# In ra)
print(df_all.to_string())

# Top-10 templates results/top_templates.csv
print("\n5. === EXPORT TOP 10 TEMPLATES ===")
results_dir = "/content/drive/MyDrive/AIOps/w1/d2/results"
os.makedirs(results_dir, exist_ok=True)

df_top10 = df_all.head(10)
export_path = os.path.join(results_dir, "top_templates.csv")

# Xuất ra CSV
df_top10.to_csv(export_path, index=False)
print(f"   -> Đã lưu thành công Top 10 vào: {export_path}")

1. Tổng số dòng trong file log gốc: 2,000 dòng

2. TUNING drain_sim_th
   - Ngưỡng sim_th = 0.3 -> Số templates tạo ra: 73
   - Ngưỡng sim_th = 0.5 -> Số templates tạo ra: 151
   - Ngưỡng sim_th = 0.7 -> Số templates tạo ra: 1459

   => LỰA CHỌN: Chọn sim_th = 0.5.
   (Lý do: 0.3 quá thấp sẽ gộp sai các lỗi khác nhau thành 1, 0.7 quá cao sẽ tách ra quá nhiều template vụn vặt làm nhiễu dữ liệu. 0.5 là mức cân bằng tốt nhất cho tập BGL).

3. PARSE LOG VỚI sim_th = 0.5
   -> Đã parse xong! Tạo ra được 151 templates unique.

4. LIỆT KÊ TẤT CẢ TEMPLATES
    template_id                                                                                                                                                                                                                                                                                                                                                                                                                                              

Phase 2: Anomaly Detection trên Log (Time Series & 3σ)

In [12]:
import numpy as np

# Time Series (window 30 phút)
df = pd.DataFrame(parsed_data)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')

# Gom nhóm mỗi 30 phút và đếm số lượng từng template_id
ts = df.groupby([pd.Grouper(key='datetime', freq='30T'), 'template_id']).size()
ts = ts.unstack(fill_value=0)

print("TEMPLATE COUNT TIME SERIES (Preview)")
print(ts.head())

# Áp dụng 3-Sigma (3σ) Detect Spike
print("\nKẾT QUẢ ANOMALY DETECTION (3σ)")
for col in ts.columns:
    counts = ts[col]
    mean = counts.mean()
    std = counts.std()

    threshold = mean + 3 * std
    anomalies = counts[(counts > threshold) & (counts > 0)]

    if not anomalies.empty:
        tmpl_str = next(c.get_template() for c in clusters if c.cluster_id == col)
        print(f"\n[!] Template T-{col} tăng vọt (Ngưỡng 3σ: {threshold:.2f}):")
        print(f"    Nội dung: {tmpl_str}")
        for time_idx, count_val in anomalies.items():
            print(f"    -> Lúc {time_idx}: Xuất hiện {count_val} lần")

# Truy vết template mới xuất hiện khi nào
print("\nKẾT QUẢ TRUY VẾT TEMPLATE MỚI XUẤT HIỆN")
# Tìm thời điểm (index) nhỏ nhất mà giá trị đếm > 0 cho từng template
first_appearances = ts.apply(lambda col: col[col > 0].index.min()).dropna().sort_values()

# In ra 5 template mới nhất vừa xuất hiện
newest_templates = first_appearances.tail(5)
for col, time_seen in newest_templates.items():
    tmpl_str = next(c.get_template() for c in clusters if c.cluster_id == col)
    print(f"\n[+] Template T-{col} LẦN ĐẦU xuất hiện lúc: {time_seen}")
    print(f"    Nội dung: {tmpl_str}")

TEMPLATE COUNT TIME SERIES (Preview)
template_id          1    2    3    4    5    6    7    8    9    10   ...  \
datetime                                                               ...   
2005-06-03 22:30:00    4    0    0    0    0    0    0    0    0    0  ...   
2005-06-03 23:30:00    0    3    0    0    0    0    0    0    0    0  ...   
2005-06-04 01:00:00    0    0    1    0    0    0    0    0    0    0  ...   
2005-06-04 07:00:00    0    0    0    2    0    0    0    0    0    0  ...   
2005-06-05 03:00:00    0    0    1    0    0    0    0    0    0    0  ...   

template_id          142  143  144  145  146  147  148  149  150  151  
datetime                                                               
2005-06-03 22:30:00    0    0    0    0    0    0    0    0    0    0  
2005-06-03 23:30:00    0    0    0    0    0    0    0    0    0    0  
2005-06-04 01:00:00    0    0    0    0    0    0    0    0    0    0  
2005-06-04 07:00:00    0    0    0    0    0    0    0  

/tmp/ipykernel_6406/761810496.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  ts = df.groupby([pd.Grouper(key='datetime', freq='30T'), 'template_id']).size()


Phase 3: Embedding + Cross-signal

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF & Tính Similarity Matrix ---
# Lấy danh sách text của tất cả templates
templates_text = [c.get_template() for c in clusters]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(templates_text)
sim_matrix = cosine_similarity(tfidf_matrix)

print("TÌM TEMPLATE TƯƠNG ĐỒNG")
# Thử xem Template 0
target_idx = 0
print(f"Template gốc: {templates_text[target_idx]}")

# Lấy điểm giống nhau của template 0 với tất cả template khác
sim_scores = list(enumerate(sim_matrix[target_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:4]

print("Top 3 template giống nhất:")
for idx, score in sim_scores:
    print(f"  - (Score: {score:.2f}) {templates_text[idx]}")

# Inject dòng log lạ
print("\nINJECT LOG LẠ")
fake_log = "- 1117838580 2005.06.03 ROOT-NODE 2005-06-03 FATAL SYSTEM HACKED BY ANONYMOUS AT IP 192.168.1.99"

# Ghi nhận số lượng cluster TRƯỚC KHI inject
cluster_count_before = len(miner.drain.clusters)

# Đưa log lạ vào hệ thống
result_fake = miner.add_log_message(fake_log)

if result_fake["change_type"] != "none":
    print("   CẢNH BÁO: Phát hiện Template hoàn toàn mới!")
    print(f"   ID Mới: {result_fake['cluster_id']}")
    print(f"   Nội dung Template: {result_fake['template_mined']}")
else:
    print("Dòng log này bị gom vào một nhóm cũ.")

TÌM TEMPLATE TƯƠNG ĐỒNG
Template gốc: - <*> <*> <*> <*> <*> RAS KERNEL INFO instruction cache parity error corrected
Top 3 template giống nhất:
  - (Score: 0.27) - <*> <*> <*> <*> <*> RAS KERNEL INFO total of <*> ddr error(s) detected and corrected
  - (Score: 0.26) - <*> 2005.10.17 R02-M0-N4-C:J04-U11 <*> R02-M0-N4-C:J04-U11 RAS KERNEL INFO data cache search parity error detected. attempting to correct
  - (Score: 0.26) - <*> 2005.06.14 <*> <*> <*> RAS KERNEL FATAL guaranteed <*> cache block <*>

INJECT LOG LẠ
   CẢNH BÁO: Phát hiện Template hoàn toàn mới!
   ID Mới: 152
   Nội dung Template: - 1117838580 2005.06.03 ROOT-NODE 2005-06-03 FATAL SYSTEM HACKED BY ANONYMOUS AT IP 192.168.1.99
